# Extracción de datos

In [35]:
from pykml import parser

def extraer_datos_kml(ruta_archivo):
    with open(ruta_archivo, 'rt', encoding='utf-8') as f:
        # Cargamos el archivo completo
        root = parser.parse(f).getroot()
    
    puntos = []
    
    # Esta búsqueda encuentra todos los 'Placemark' en cualquier parte del archivo
    # eliminando el problema del AttributeError por jerarquía
    for pm in root.xpath('.//*[local-name()="Placemark"]'):
        try:
            nombre = str(pm.name).strip()
            # KML usa: longitud, latitud, altitud
            coords_raw = str(pm.Point.coordinates).strip().split(',')
            
            puntos.append({
                'nombre': nombre,
                'lat': float(coords_raw[1]),
                'lng': float(coords_raw[0])
            })
        except AttributeError:
            # Si encuentra un Placemark que no tiene punto (ej. una ruta), lo ignora
            continue
            
    return puntos

# --- PRUEBA DE CARGA ---
seleccion_fincas = extraer_datos_kml('PUNTOS.kml')
print(f"✅ Éxito: Se han cargado {len(seleccion_fincas)} puntos.")


✅ Éxito: Se han cargado 8 puntos.


# CONEXION CON EL MOTOR DE MAPAS

In [14]:
import requests

def obtener_matriz_distancias(puntos):
    # Formateamos coordenadas para la URL de OSRM (lon,lat;lon,lat...)
    coords_str = ";".join([f"{p['lng']},{p['lat']}" for p in puntos])
    
    url = f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance"
    
    try:
        response = requests.get(url)
        data = response.json()
        if data['code'] == 'Ok':
            # Convertimos de metros a kilómetros
            return [[round(d / 1000, 2) for d in fila] for fila in data['distances']]
        return None
    except Exception as e:
        print(f"Error de conexión OSRM: {e}")
        return None

# VISUALIZACION DE MATRIZ DE DISTANCIAS

In [41]:
import requests
import pandas as pd
import itertools
import time

def obtener_matriz_osrm_metros(puntos):
    # Formato: lon,lat;lon,lat
    coords_str = ";".join([f"{p['lng']},{p['lat']}" for p in puntos])
    url = f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance"
    
    response = requests.get(url).json()
    if response['code'] == 'Ok':
        # Esta es la matriz cruda en METROS
        matriz_metros = response['distances']
        return matriz_metros
    return None

# --- PROCESAMIENTO ---
# Supongamos que 'seleccion_fincas' es la lista de puntos del KML
matriz_m = obtener_matriz_osrm_metros(seleccion_fincas)

# Convertimos a KM para los algoritmos y visualización
matriz_km = [[round(d / 1000, 2) for d in fila] for fila in matriz_m]

print("\n" + "="*60)
print("MATRIZ CRUDA (METROS) RECIBIDA DEL MODELO OSRM")
print("="*60)

df_m = pd.DataFrame(matriz_m)
print(df_m.to_string(index=True, header=True))

print("\n" + "="*60)
print("MATRIZ CONVERTIDA A KM")
print("="*60)

df_km = pd.DataFrame(matriz_km)
print(df_km.to_string(index=True, header=True))


MATRIZ CRUDA (METROS) RECIBIDA DEL MODELO OSRM
          0         1         2         3         4         5         6         7
0       0.0   26787.6   69775.3   74600.8   35988.2   79966.5  167826.5   29366.8
1   26983.0       0.0   96287.1   54133.5   62500.0  106478.4  194338.3    3775.5
2   69473.2   93540.0       0.0  141353.1   37259.5   33179.0  121039.0   96119.2
3   74796.1   54133.5  144100.3       0.0  110313.2  154291.5  242151.5   50358.1
4   35686.1   59752.9   37259.5  107566.0       0.0   47450.7  135310.7   62332.1
5   79664.4  103731.2   33179.0  151544.3   47450.7       0.0   87954.2  106310.4
6  167524.4  191591.1  121039.0  239404.3  135310.7   87954.2       0.0  194170.3
7   29562.2    3775.5   98866.3   50358.0   65079.2  109057.6  196917.5       0.0

MATRIZ CONVERTIDA A KM
        0       1       2       3       4       5       6       7
0    0.00   26.79   69.78   74.60   35.99   79.97  167.83   29.37
1   26.98    0.00   96.29   54.13   62.50  106.48  194.34 

In [37]:
def mostrar_matriz_nombres(matriz_km, puntos):
    nombres = [p['nombre'] for p in puntos]
    df = pd.DataFrame(matriz_km, index=nombres, columns=nombres)
    
    print("\n" + "="*60)
    print("MATRIZ DE DISTANCIAS POR CARRETERA (KILÓMETROS)")
    print("="*60)
    print(df.to_string())
    print("="*60)

mostrar_matriz_nombres(matriz_km, seleccion_fincas)


MATRIZ DE DISTANCIAS POR CARRETERA (KILÓMETROS)
                     Empresa 1  Finca 1(Soloco)  Finca 2(La Jalca)  Finca 3(Chiliquin)  Finca 4(Magdalena)  Finca 5(Leymebamba)  Finca 6(Balsas)  Finca 7(Daguas)
Empresa 1                 0.00            26.79              69.78               74.60               35.99                79.97           167.83            29.37
Finca 1(Soloco)          26.98             0.00              96.29               54.13               62.50               106.48           194.34             3.78
Finca 2(La Jalca)        69.47            93.54               0.00              141.35               37.26                33.18           121.04            96.12
Finca 3(Chiliquin)       74.80            54.13             144.10                0.00              110.31               154.29           242.15            50.36
Finca 4(Magdalena)       35.69            59.75              37.26              107.57                0.00                47.45           135

# ALGORITMOS

In [38]:
# --- FUNCIONES DE ALGORITMOS ---
def algoritmo_voraz(matriz, inicio):
    n = len(matriz); visitados = [False] * n; ruta = [inicio]
    visitados[inicio] = True; dist = 0; actual = inicio
    for _ in range(n - 1):
        siguiente = -1; d_min = float('inf')
        for i in range(n):
            if not visitados[i] and matriz[actual][i] < d_min:
                d_min = matriz[actual][i]; siguiente = i
        visitados[siguiente] = True; ruta.append(siguiente)
        dist += d_min; actual = siguiente
    dist += matriz[actual][inicio]; ruta.append(inicio)
    return ruta, dist

def algoritmo_fuerza_bruta(matriz, inicio):
    n = len(matriz); nodos = [i for i in range(n) if i != inicio]
    mejor_dist = float('inf'); mejor_ruta = []
    for perm in itertools.permutations(nodos):
        ruta = [inicio] + list(perm) + [inicio]
        d = sum(matriz[ruta[i]][ruta[i+1]] for i in range(len(ruta)-1))
        if d < mejor_dist: mejor_dist = d; mejor_ruta = ruta
    return mejor_ruta, mejor_dist

def algoritmo_dinamico(matriz, inicio):
    n = len(matriz); memo = {}
    def visitar(mask, pos):
        if mask == (1 << n) - 1: return matriz[pos][inicio], [pos, inicio]
        if (mask, pos) in memo: return memo[(mask, pos)]
        res = float('inf'); camino = []
        for sig in range(n):
            if (mask >> sig) & 1 == 0:
                d, c = visitar(mask | (1 << sig), sig)
                if matriz[pos][sig] + d < res:
                    res = matriz[pos][sig] + d; camino = [pos] + c
        memo[(mask, pos)] = (res, camino)
        return res, camino
    return visitar(1 << inicio, inicio)[1], visitar(1 << inicio, inicio)[0]

# --- EJECUCIÓN Y COMPARATIVA ---
inicio_idx = 0
res_v = algoritmo_voraz(matriz_km, inicio_idx)
res_fb = algoritmo_fuerza_bruta(matriz_km, inicio_idx)
res_pd = algoritmo_dinamico(matriz_km, inicio_idx)

print(f"\n{'ALGORITMO':<20} | {'DISTANCIA':<12} | {'RUTA'}")
print("-" * 70)
print(f"{'Voraz':<20} | {res_v[1]:<12.2f} | {res_v[0]}")
print(f"{'Fuerza Bruta':<20} | {res_fb[1]:<12.2f} | {res_fb[0]}")
print(f"{'Prog. Dinámica':<20} | {res_pd[1]:<12.2f} | {res_pd[0]}")

# RECOMENDACIÓN FINAL
nombres_optimos = [seleccion_fincas[i]['nombre'] for i in res_pd[0]]
print(f"\n>>> RECOMENDACIÓN DE RUTA ÓPTIMA:")
print(f"La mejor ruta es mediante Programación Dinámica con {res_pd[1]:.2f} km.")
print(f"Orden sugerido: {' -> '.join(nombres_optimos)}")


ALGORITMO            | DISTANCIA    | RUTA
----------------------------------------------------------------------
Voraz                | 517.15       | [0, 1, 7, 3, 4, 2, 5, 6, 0]
Fuerza Bruta         | 514.90       | [0, 4, 2, 5, 6, 3, 7, 1, 0]
Prog. Dinámica       | 514.90       | [0, 4, 2, 5, 6, 3, 7, 1, 0]

>>> RECOMENDACIÓN DE RUTA ÓPTIMA:
La mejor ruta es mediante Programación Dinámica con 514.90 km.
Orden sugerido: Empresa 1 -> Finca 4(Magdalena) -> Finca 2(La Jalca) -> Finca 5(Leymebamba) -> Finca 6(Balsas) -> Finca 3(Chiliquin) -> Finca 7(Daguas) -> Finca 1(Soloco) -> Empresa 1


**MODELO DE GOOGLE**

In [39]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def resolver_google_optimizacion(matriz_distancias, punto_inicio):
    # Multiplicamos por 100 para manejar decimales como enteros (requerido por OR-Tools)
    matriz_int = [[int(val * 100) for val in fila] for fila in matriz_distancias]
    
    manager = pywrapcp.RoutingIndexManager(len(matriz_int), 1, punto_inicio)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return matriz_int[manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # CONFIGURACIÓN DE LAS DOS ETAPAS
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    
    # Etapa 1: Solución Inicial (Voraz - Arco más barato)
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    
    # Etapa 2: Búsqueda Local (Optimización metaheurística)
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = 2 # Tiempo límite para buscar mejoras

    solution = routing.SolveWithParameters(search_parameters)
    
    if solution:
        ruta = []
        index = routing.Start(0)
        while not routing.IsEnd(index):
            ruta.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        ruta.append(manager.IndexToNode(index))
        distancia = solution.ObjectiveValue() / 100.0
        return ruta, distancia
    return None, 0.0

**COMPARACION FINAL**

In [40]:
import time

def imprimir_reporte_tesis(matriz_km, puntos, inicio_idx):
    print(f"\n{'='*105}")
    print(f"{'REPORTE DE OPTIMIZACIÓN LOGÍSTICA: COMPARATIVA DE ALGORITMOS':^105}")
    print(f"{'='*105}")
    print(f"{'TIPO':<12} | {'ALGORITMO':<28} | {'DISTANCIA (KM)':<18} | {'TIEMPO (SEG)':<15} | {'ESTADO'}")
    print(f"{'-'*105}")

    # --- 1. ALGORITMOS MANUALES ---
    
    # Voraz
    t_init = time.time()
    r_vo, d_vo = algoritmo_voraz(matriz_km, inicio_idx)
    t_vo = time.time() - t_init
    print(f"{'MANUAL':<12} | {'Algoritmo Voraz':<28} | {d_vo:<18.2f} | {t_vo:<15.6f} | Heurístico")

    # Fuerza Bruta
    t_init = time.time()
    r_fb, d_fb = algoritmo_fuerza_bruta(matriz_km, inicio_idx)
    t_fb = time.time() - t_init
    print(f"{'MANUAL':<12} | {'Fuerza Bruta':<28} | {d_fb:<18.2f} | {t_fb:<15.6f} | Óptimo Global")

    # Programación Dinámica
    t_init = time.time()
    r_pd, d_pd = algoritmo_dinamico(matriz_km, inicio_idx)
    t_pd = time.time() - t_init
    print(f"{'MANUAL':<12} | {'Programación Dinámica':<28} | {d_pd:<18.2f} | {t_pd:<15.6f} | Óptimo Global")

    print(f"{'-'*105}")

    # --- 2. OPTIMIZACIÓN INDUSTRIAL ---
    
    t_init = time.time()
    r_go, d_go = resolver_google_optimizacion(matriz_km, inicio_idx)
    t_go = time.time() - t_init
    print(f"{'GOOGLE':<12} | {'Voraz + Búsqueda Local (GLS)':<28} | {d_go:<18.2f} | {t_go:<15.6f} | Profesional")

    print(f"{'='*105}")

    # --- ANÁLISIS FINAL ---
    print(f"\nRESUMEN DE RESULTADOS:")
    print(f"• La mejor ruta encontrada mide {d_pd:.2f} km.")
    print(f"• El algoritmo manual más preciso fue la Programación Dinámica.")
    print(f"• Google OR-Tools logró el mismo óptimo en un tiempo competitivo usando metaheurísticas.")
    
    nombres_ruta = [puntos[i]['nombre'] for i in r_pd]
    print(f"\n>>> SECUENCIA ÓPTIMA FINAL (Basada en Dinámica/Google):")
    print(f"    {' -> '.join(nombres_ruta)}")

# Ejecutar el reporte
imprimir_reporte_tesis(matriz_km, seleccion_fincas, 0)


                      REPORTE DE OPTIMIZACIÓN LOGÍSTICA: COMPARATIVA DE ALGORITMOS                       
TIPO         | ALGORITMO                    | DISTANCIA (KM)     | TIEMPO (SEG)    | ESTADO
---------------------------------------------------------------------------------------------------------
MANUAL       | Algoritmo Voraz              | 517.15             | 0.000168        | Heurístico
MANUAL       | Fuerza Bruta                 | 514.90             | 0.018113        | Óptimo Global
MANUAL       | Programación Dinámica        | 514.90             | 0.003286        | Óptimo Global
---------------------------------------------------------------------------------------------------------
GOOGLE       | Voraz + Búsqueda Local (GLS) | 514.90             | 2.092424        | Profesional

RESUMEN DE RESULTADOS:
• La mejor ruta encontrada mide 514.90 km.
• El algoritmo manual más preciso fue la Programación Dinámica.
• Google OR-Tools logró el mismo óptimo en un tiempo competitivo us